# 📊 財務分析ツール
Google Drive のファイルをブラウザから選んで財務指標を自動分析します。

**使い方:** 上から順に ▶️ を押してください。ファイルマネージャーで番号を入力するだけでOKです。

## ① ライブラリをインストール（初回のみ）

In [ ]:
!pip install pymupdf openpyxl python-docx -q
print('✅ インストール完了')

## ② 分析コードを読み込む

In [ ]:
from dataclasses import dataclass
from typing import Optional, Dict, List
import re, os
from pathlib import Path

@dataclass
class IncomeStatement:
    revenue: float
    cost_of_goods_sold: float
    gross_profit: Optional[float] = None
    selling_expenses: float = 0.0
    general_admin_expenses: float = 0.0
    operating_expenses: Optional[float] = None
    operating_income: Optional[float] = None
    non_operating_income: float = 0.0
    non_operating_expenses: float = 0.0
    ordinary_income: Optional[float] = None
    extraordinary_income: float = 0.0
    extraordinary_losses: float = 0.0
    income_before_tax: Optional[float] = None
    income_tax: float = 0.0
    net_income: Optional[float] = None
    period: str = ''
    def __post_init__(self):
        if self.gross_profit is None: self.gross_profit = self.revenue - self.cost_of_goods_sold
        if self.operating_expenses is None: self.operating_expenses = self.selling_expenses + self.general_admin_expenses
        if self.operating_income is None: self.operating_income = self.gross_profit - self.operating_expenses
        if self.ordinary_income is None: self.ordinary_income = self.operating_income + self.non_operating_income - self.non_operating_expenses
        if self.income_before_tax is None: self.income_before_tax = self.ordinary_income + self.extraordinary_income - self.extraordinary_losses
        if self.net_income is None: self.net_income = self.income_before_tax - self.income_tax

KEYWORD_MAP = {
    '売上高':'revenue','売上':'revenue','収益合計':'revenue','営業収益':'revenue',
    '売上原価':'cost_of_goods_sold','原価合計':'cost_of_goods_sold','製造原価':'cost_of_goods_sold',
    '売上総利益':'gross_profit','粗利':'gross_profit','粗利益':'gross_profit',
    '販売費及び一般管理費':'selling_and_admin','販管費':'selling_and_admin',
    '販売費':'selling_expenses','一般管理費':'general_admin_expenses','管理費':'general_admin_expenses',
    '営業利益':'operating_income','営業損益':'operating_income',
    '営業外収益':'non_operating_income',
    '営業外費用':'non_operating_expenses','支払利息':'non_operating_expenses',
    '経常利益':'ordinary_income','経常損益':'ordinary_income',
    '特別利益':'extraordinary_income',
    '特別損失':'extraordinary_losses','特別費用':'extraordinary_losses',
    '税引前当期純利益':'income_before_tax','税引前利益':'income_before_tax',
    '法人税':'income_tax','法人税等':'income_tax',
    '当期純利益':'net_income','当期純損失':'net_income','当期利益':'net_income','純利益':'net_income',
    '営業損失':'operating_income',
    '経常損失':'ordinary_income',
    '税引前当期純損失':'income_before_tax',
    '受取利息':'non_operating_income',
    '法人税、住民税及び事業税':'income_tax',
    '売上収入':'revenue',
}

def _parse_number(text):
    if not text: return None
    t = str(text).strip().replace(',','').replace('，','').replace(' ','')
    if not t or t in ('-','―','—','－'): return 0.0
    neg = False
    if t.startswith(('△','▲','▽')): neg,t = True,t[1:]
    elif t.startswith('(') and t.endswith(')'): neg,t = True,t[1:-1]
    elif t.startswith('（') and t.endswith('）'): neg,t = True,t[1:-1]
    try: v=float(t); return -v if neg else v
    except: return None

def _match_keyword(label):
    label = label.strip()
    if label in KEYWORD_MAP: return KEYWORD_MAP[label]
    for kw in sorted(KEYWORD_MAP.keys(), key=len, reverse=True):
        if kw in label: return KEYWORD_MAP[kw]
    return None

def _build_income(fields, period):
    if 'selling_and_admin' in fields and 'selling_expenses' not in fields:
        c=fields.pop('selling_and_admin'); fields['selling_expenses']=c*0.5; fields['general_admin_expenses']=c*0.5
    else: fields.pop('selling_and_admin',None)
    def g(k): return fields.get(k,0.0)
    kw=dict(period=period,revenue=g('revenue'),cost_of_goods_sold=g('cost_of_goods_sold'),
            selling_expenses=g('selling_expenses'),general_admin_expenses=g('general_admin_expenses'),
            non_operating_income=g('non_operating_income'),non_operating_expenses=g('non_operating_expenses'),
            extraordinary_income=g('extraordinary_income'),extraordinary_losses=g('extraordinary_losses'),income_tax=g('income_tax'))
    for k in ('gross_profit','operating_income','ordinary_income','income_before_tax','net_income'):
        if k in fields: kw[k]=fields[k]
    return IncomeStatement(**kw)

TOTAL_KEYWORDS = ('合計','累計','年計','通期','決算')

def _resolve_total_col(header, month_re):
    for i,h in enumerate(header):
        if any(k in h for k in TOTAL_KEYWORDS): return i
    mc=[i for i,h in enumerate(header) if month_re.search(h)]
    return max(mc) if len(mc)>=3 else None

def load_excel(path, period='', unit=1.0, sheet_name=None):
    import openpyxl
    wb=openpyxl.load_workbook(path,read_only=True,data_only=True)
    ws=wb[wb.sheetnames[0]]
    for name in wb.sheetnames:
        if sheet_name and name==sheet_name: ws=wb[name]; break
        elif not sheet_name and any(k in name for k in ('損益','PL','P&L','売上')): ws=wb[name]; break
    rows=[list(r) for r in ws.iter_rows(values_only=True)]; wb.close()
    if not period:
        for row in rows[:5]:
            for c in row:
                m=re.search(r'(\d{4}年\d{1,2}月期)',str(c or ''))
                if m: period=m.group(1); break
    month_re=re.compile(r'\d{1,2}月')
    header_idx,header=0,[str(c or '').strip() for c in rows[0]]
    for i,row in enumerate(rows[:20]):
        strs=[str(c or '').strip() for c in row]
        if sum(1 for s in strs if month_re.search(s))>=3 or any(k in ' '.join(strs) for k in ('合計','売上高','勘定科目')):
            header_idx,header=i,strs; break
    total_col=_resolve_total_col(header,month_re)
    fields={}
    if total_col is not None:
        for row in rows[header_idx+1:]:
            label=str(row[0] or '').strip(); f=_match_keyword(label)
            if f and total_col<len(row):
                v=_parse_number(row[total_col])
                if v is not None: fields.setdefault(f,v*unit)
    else:
        for row in rows[header_idx+1:]:
            label=str(row[0] or '').strip(); f=_match_keyword(label)
            if f:
                for c in reversed(row[1:]):
                    v=_parse_number(c)
                    if v is not None: fields.setdefault(f,v*unit); break
    return _build_income(fields,period)

def load_pdf(path, period='', unit=1.0):
    import fitz
    all_rows=[]
    with fitz.open(path) as doc:
        for page in doc:
            words=page.get_text('words'); pw=page.rect.width
            lw,nw={},{}
            for x0,y0,x1,y1,text,*_ in words:
                yk=int(y0/6)*6
                (lw if x0<pw*0.4 else nw).setdefault(yk,[]).append(text.strip())
            if not period:
                m=re.search(r'(\d{4}年\d{1,2}月期)',page.get_text())
                if m: period=m.group(1)
            for yk in sorted(lw):
                label=' '.join(lw[yk]); nums=nw.get(yk,[])
                if not nums:
                    for dy in (6,12,-6,-12):
                        nums=nw.get(yk+dy,[])
                        if nums: break
                all_rows.append([label, nums[-1] if nums else ''])
    fields={}
    for row in all_rows:
        f=_match_keyword(row[0])
        if f:
            v=_parse_number(row[1])
            if v is not None: fields.setdefault(f,v*unit)
    return _build_income(fields,period)

def load_word(path, period='', unit=1.0):
    import docx
    doc=docx.Document(path)
    if not period:
        for p in doc.paragraphs[:10]:
            m=re.search(r'(\d{4}年\d{1,2}月期)',p.text)
            if m: period=m.group(1); break
    fields={}; month_re=re.compile(r'\d{1,2}月')
    for table in doc.tables:
        rows=[[c.text.strip() for c in row.cells] for row in table.rows]
        if not rows: continue
        header=rows[0]; total_col=_resolve_total_col(header,month_re)
        _all_fields = set(KEYWORD_MAP.values()) - {'selling_and_admin'}
        for row in rows[1:] if total_col is not None else rows:
            if len(fields) >= len(_all_fields): break
            label=row[0]; f=_match_keyword(label)
            if f:
                cells=[row[total_col]] if total_col is not None and total_col<len(row) else row[1:]
                for c in reversed(cells):
                    v=_parse_number(c)
                    if v is not None: fields.setdefault(f,v*unit); break
    return _build_income(fields,period)

def load_file(path, period='', unit=1.0, sheet_name=None):
    ext=Path(path).suffix.lower()
    if ext=='.pdf': return load_pdf(path,period,unit)
    if ext in ('.xlsx','.xls','.xlsm'): return load_excel(path,period,unit,sheet_name)
    if ext=='.docx': return load_word(path,period,unit)
    raise ValueError(f'非対応形式: {ext}')

def analyze_and_print(is_, company_name=''):
    sep='='*55
    print(sep)
    print(f'  財務分析レポート: {company_name}')
    print(f'  対象期間: {is_.period}')
    print(sep)
    def pct(n,d): return (n/d*100) if d else 0.0
    print('\n【主要数値】')
    print(f'  売上高:       {is_.revenue:>15,.0f} 円')
    print(f'  売上原価:     {is_.cost_of_goods_sold:>15,.0f} 円')
    print(f'  売上総利益:   {is_.gross_profit:>15,.0f} 円')
    print(f'  営業利益:     {is_.operating_income:>15,.0f} 円')
    print(f'  経常利益:     {is_.ordinary_income:>15,.0f} 円')
    print(f'  当期純利益:   {is_.net_income:>15,.0f} 円')
    print('\n【収益性指標】')
    gm=pct(is_.gross_profit,is_.revenue); om=pct(is_.operating_income,is_.revenue); nm=pct(is_.net_income,is_.revenue)
    print(f'  売上総利益率: {gm:6.2f}%  ','★★★' if gm>=50 else '★★' if gm>=30 else '★')
    print(f'  営業利益率:   {om:6.2f}%  ','★★★' if om>=15 else '★★' if om>=5 else '★')
    print(f'  純利益率:     {nm:6.2f}%  ','★★★' if nm>=10 else '★★' if nm>=3 else '★')
    print('\n【評価コメント】')
    if gm>=50: print('  ・粗利率50%超 — 非常に高い価格競争力があります')
    elif gm>=30: print('  ・粗利率は良好な水準です')
    else: print('  ・粗利率が低め。原価管理の見直しを検討してください')
    if om>=15: print('  ・営業利益率15%超 — 優秀な収益力です')
    elif om>=5: print('  ・営業利益率は安定した水準です')
    elif om>=0: print('  ・営業利益率が低め。販管費の削減を検討してください')
    else: print('  ・営業損失が発生しています。早急な改善が必要です')
    print(sep)

print('✅ 分析コード読み込み完了')

## ③ Google Drive に接続
ポップアップが出たら **「Googleドライブに接続」** をタップしてください

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ 接続完了')

## ④ 📁 ファイルマネージャー
ドライブのファイルを一覧表示して番号で選択できます

In [ ]:
import os, re
from pathlib import Path
from IPython.display import display, HTML

# ─── 設定 ───────────────────────────────────
SEARCH_ROOT   = '/content/drive/MyDrive'  # 検索起点（変更可）
SEARCH_DEPTH  = 3                          # フォルダの深さ（1=直下のみ）
SUPPORTED_EXT = {'.pdf', '.xlsx', '.xls', '.xlsm', '.docx'}
# ────────────────────────────────────────────

EXT_ICON = {'.pdf': '📄', '.xlsx': '📊', '.xls': '📊', '.xlsm': '📊', '.docx': '📝'}

def scan_drive(root, max_depth):
    found = []
    def _scan(path, depth):
        if depth > max_depth: return
        try:
            for entry in sorted(os.scandir(path), key=lambda e: (e.is_file(), e.name)):
                if entry.is_dir():
                    _scan(entry.path, depth + 1)
                elif Path(entry.name).suffix.lower() in SUPPORTED_EXT:
                    found.append(entry.path)
        except PermissionError:
            pass
    _scan(root, 1)
    return found

print('📁 ファイルを検索中...')
files = scan_drive(SEARCH_ROOT, SEARCH_DEPTH)

if not files:
    print('⚠️ 対応ファイルが見つかりませんでした。')
    print(f'   検索場所: {SEARCH_ROOT}')
    print(f'   対応形式: {SUPPORTED_EXT}')
else:
    print(f'\n{len(files)} 件のファイルが見つかりました:\n')
    header = f'{'番号':>4}  {'種類':2}  ファイル名（フォルダ）'
    print(header)
    print('-' * 70)
    for i, fpath in enumerate(files, 1):
        p = Path(fpath)
        ext = p.suffix.lower()
        icon = EXT_ICON.get(ext, '📎')
        rel = fpath.replace(SEARCH_ROOT + '/', '')
        display_name = rel if len(rel) <= 60 else '...' + rel[-57:]
        print(f'{i:>4}  {icon}   {display_name}')
    print('-' * 70)
    print('\n👇 次のセルで番号を入力して分析を実行してください')

# グローバルに保持
_fm_files = files

## ⑤ 番号を入力して分析実行
上の一覧で確認した番号を入力してください

In [ ]:
# ==============================
# 番号と設定を入力してください
# ==============================
FILE_NUMBER  = 1           # ← 上の一覧の番号
COMPANY_NAME = '株式会社〇〇'  # ← 企業名
PERIOD       = ''          # ← 会計期間（空欄=自動検出）
UNIT         = 1           # ← 1=円, 1000=千円, 1000000=百万円
SHEET_NAME   = None        # ← Excelシート名（None=自動）
# ==============================

if not globals().get('_fm_files'):
    print('⚠️ 先にセル④を実行してファイル一覧を取得してください')
elif FILE_NUMBER < 1 or FILE_NUMBER > len(_fm_files):
    print(f'⚠️ 番号は 1〜{len(_fm_files)} の範囲で入力してください')
else:
    selected = _fm_files[FILE_NUMBER - 1]
    print(f'選択したファイル: {Path(selected).name}')
    print(f'パス: {selected}\n')
    try:
        is_ = load_file(selected, period=PERIOD, unit=UNIT, sheet_name=SHEET_NAME)
        analyze_and_print(is_, company_name=COMPANY_NAME)
    except Exception as e:
        print(f'❌ エラー: {e}')
        print('\nヒント: セル⑦のデバッグセルを実行してファイル内容を確認してください')

## ⑥ 複数ファイルをまとめて分析

In [ ]:
# ④の一覧番号で複数ファイルを指定
SELECTIONS = [
    {'no': 1, 'company': '会社A', 'unit': 1},
    {'no': 2, 'company': '会社B', 'unit': 1000},
    {'no': 3, 'company': '会社C', 'unit': 1},
]

for sel in SELECTIONS:
    no = sel['no']
    if no < 1 or no > len(_fm_files):
        print(f'⚠️ 番号 {no} は範囲外です'); continue
    fpath = _fm_files[no - 1]
    print(f'\n[{no}] {Path(fpath).name}')
    try:
        is_ = load_file(fpath, unit=sel.get('unit',1))
        analyze_and_print(is_, company_name=sel['company'])
    except Exception as e:
        print(f'  ❌ エラー: {e}')

## ⑦ 🔍 デバッグ（うまく読み込めない場合）

In [ ]:
# ④の番号を入れて実行
DEBUG_FILE_NUMBER = 1

if not globals().get('_fm_files'):
    print('先にセル④を実行してください')
else:
    fpath = _fm_files[DEBUG_FILE_NUMBER - 1]
    ext = Path(fpath).suffix.lower()
    print(f'ファイル: {fpath}\n')

    if ext in ('.xlsx', '.xls', '.xlsm'):
        import openpyxl
        wb = openpyxl.load_workbook(fpath, read_only=True, data_only=True)
        print('シート一覧:', wb.sheetnames)
        ws = wb[wb.sheetnames[0]]
        print('\n先頭15行:')
        for i, row in enumerate(ws.iter_rows(values_only=True)):
            if i >= 15: break
            print(f'  行{i+1:2}: {[str(c)[:15] if c else "" for c in row]}')
        wb.close()

    elif ext == '.pdf':
        import fitz
        with fitz.open(fpath) as doc:
            print('ページ数:', len(doc))
            print('\n1ページ目テキスト（先頭30行）:')
            lines = doc[0].get_text().split('\n')
            for i, line in enumerate(lines[:30]):
                if line.strip(): print(f'  {line}')

    elif ext == '.docx':
        import docx
        doc = docx.Document(fpath)
        print(f'段落数: {len(doc.paragraphs)}, 表数: {len(doc.tables)}')
        if doc.tables:
            print('\n最初の表（先頭10行）:')
            for i, row in enumerate(doc.tables[0].rows):
                if i >= 10: break
                print(f'  {[c.text[:15] for c in row.cells]}')

## ⑧ 📄 ドキュメントをダウンロード
ドキュメント（mdファイル）を ZIP にまとめてダウンロードします

In [ ]:
# ドキュメントファイルをダウンロード
# このセルを実行すると docs/ フォルダが作成され、各mdファイルをダウンロードできます
import os
from google.colab import files

DOCS = {
    "01_models.md": "# データモデル (`financial_analysis/models.py`)\n\n財務諸表の数値を保持するデータクラス群です。  \n`__post_init__` により、省略したフィールドは自動計算されます。\n\n---\n\n## IncomeStatement（損益計算書）\n\n```python\n@dataclass\nclass IncomeStatement:\n    revenue: float                  # 売上高（必須）\n    cost_of_goods_sold: float       # 売上原価（必須）\n    gross_profit: Optional[float]   # 売上総利益（自動: revenue - COGS）\n\n    selling_expenses: float = 0.0\n    general_admin_expenses: float = 0.0\n    operating_expenses: Optional[float]  # 自動: 販売費 + 一般管理費\n\n    operating_income: Optional[float]    # 自動: 粗利 - 営業費用\n    non_operating_income: float = 0.0\n    non_operating_expenses: float = 0.0\n    ordinary_income: Optional[float]     # 自動: 営業利益 ± 営業外損益\n\n    extraordinary_income: float = 0.0\n    extraordinary_losses: float = 0.0\n    income_before_tax: Optional[float]   # 自動: 経常利益 ± 特別損益\n\n    income_tax: float = 0.0\n    net_income: Optional[float]          # 自動: 税引前利益 - 法人税等\n\n    period: str = \"\"                     # 例: \"2024年3月期\"\n```\n\n### 自動計算の流れ\n\n```\n売上高 - 売上原価 = 売上総利益\n売上総利益 - 営業費用 = 営業利益\n営業利益 ± 営業外損益 = 経常利益\n経常利益 ± 特別損益 = 税引前当期純利益\n税引前当期純利益 - 法人税等 = 当期純利益\n```\n\n---\n\n## BalanceSheet（貸借対照表）\n\n```python\n@dataclass\nclass BalanceSheet:\n    # 流動資産\n    cash_and_equivalents: float = 0.0\n    accounts_receivable: float = 0.0\n    inventory: float = 0.0\n    other_current_assets: float = 0.0\n    total_current_assets: Optional[float]       # 自動計算\n\n    # 固定資産\n    property_plant_equipment: float = 0.0\n    intangible_assets: float = 0.0\n    investments: float = 0.0\n    total_non_current_assets: Optional[float]   # 自動計算\n    total_assets: Optional[float]               # 自動計算\n\n    # 流動負債\n    accounts_payable: float = 0.0\n    short_term_debt: float = 0.0\n    other_current_liabilities: float = 0.0\n    total_current_liabilities: Optional[float]  # 自動計算\n\n    # 固定負債\n    long_term_debt: float = 0.0\n    other_non_current_liabilities: float = 0.0\n    total_non_current_liabilities: Optional[float]  # 自動計算\n    total_liabilities: Optional[float]          # 自動計算\n\n    # 純資産\n    common_stock: float = 0.0\n    retained_earnings: float = 0.0\n    other_equity: float = 0.0\n    total_equity: Optional[float]               # 自動計算\n\n    period: str = \"\"\n```\n\n---\n\n## CashFlowStatement（キャッシュフロー計算書）\n\n```python\n@dataclass\nclass CashFlowStatement:\n    # 営業CF\n    net_income: float = 0.0\n    depreciation_amortization: float = 0.0\n    changes_in_working_capital: float = 0.0\n    other_operating_cf: float = 0.0\n    operating_cash_flow: Optional[float]   # 自動計算\n\n    # 投資CF\n    capital_expenditures: float = 0.0     # 設備投資（マイナス値）\n    acquisitions: float = 0.0\n    other_investing_cf: float = 0.0\n    investing_cash_flow: Optional[float]  # 自動計算\n\n    # 財務CF\n    debt_issuance: float = 0.0\n    debt_repayment: float = 0.0           # 返済（マイナス値）\n    dividends_paid: float = 0.0           # 配当（マイナス値）\n    other_financing_cf: float = 0.0\n    financing_cash_flow: Optional[float]  # 自動計算\n\n    net_change_in_cash: Optional[float]   # 自動: 三CF合計\n\n    period: str = \"\"\n```\n\n---\n\n## FinancialData（まとめコンテナ）\n\n```python\n@dataclass\nclass FinancialData:\n    company_name: str\n    income_statement: IncomeStatement\n    balance_sheet: BalanceSheet\n    cash_flow_statement: CashFlowStatement\n    previous_income_statement: Optional[IncomeStatement] = None  # 前期（成長率算出に使用）\n    previous_balance_sheet: Optional[BalanceSheet] = None\n```\n\n---\n\n## 使用例\n\n```python\nfrom financial_analysis.models import IncomeStatement, BalanceSheet, CashFlowStatement, FinancialData\n\nis_ = IncomeStatement(\n    period=\"2024年3月期\",\n    revenue=50_000_000,\n    cost_of_goods_sold=20_000_000,\n    selling_expenses=8_000_000,\n    general_admin_expenses=4_000_000,\n    income_tax=2_700_000,\n)\n# gross_profit, operating_income, net_income は自動計算される\n\nbs = BalanceSheet(\n    cash_and_equivalents=5_000_000,\n    accounts_receivable=8_000_000,\n    # ... 省略時は0\n)\n\ncf = CashFlowStatement(\n    net_income=is_.net_income,\n    depreciation_amortization=2_000_000,\n    capital_expenditures=-3_000_000,\n)\n\ndata = FinancialData(\n    company_name=\"株式会社〇〇\",\n    income_statement=is_,\n    balance_sheet=bs,\n    cash_flow_statement=cf,\n)\n```\n",
    "02_ratios.md": "# 財務比率 (`financial_analysis/ratios.py`)\n\n`FinancialRatios` クラスが財務諸表データから各種指標を計算します。\n\n---\n\n## 計算クラス\n\n```python\nratios = FinancialRatios(\n    income_stmt,\n    balance_sheet,\n    cash_flow,\n    prev_income_stmt=None,   # 前期（成長率算出に使用）\n    prev_balance_sheet=None,\n)\n```\n\n---\n\n## 収益性指標 — `ProfitabilityRatios`\n\n| メソッド | 指標 | 計算式 |\n|----------|------|--------|\n| `gross_profit_margin` | 売上総利益率 (%) | 売上総利益 / 売上高 × 100 |\n| `operating_profit_margin` | 営業利益率 (%) | 営業利益 / 売上高 × 100 |\n| `net_profit_margin` | 純利益率 (%) | 当期純利益 / 売上高 × 100 |\n| `return_on_assets` | ROA (%) | 当期純利益 / 総資産 × 100 |\n| `return_on_equity` | ROE (%) | 当期純利益 / 純資産 × 100 |\n| `ebitda_margin` | EBITDAマージン (%) | (営業利益 + 減価償却費) / 売上高 × 100 |\n\n```python\np = ratios.profitability()\nprint(p.operating_profit_margin)  # 例: 12.5\n```\n\n---\n\n## 安全性指標 — `SafetyRatios`\n\n| メソッド | 指標 | 計算式 |\n|----------|------|--------|\n| `current_ratio` | 流動比率 (%) | 流動資産 / 流動負債 × 100 |\n| `quick_ratio` | 当座比率 (%) | (現金 + 売掛金) / 流動負債 × 100 |\n| `debt_to_equity_ratio` | 負債資本比率 (倍) | 負債合計 / 純資産合計 |\n| `equity_ratio` | 自己資本比率 (%) | 純資産 / 総資産 × 100 |\n| `interest_coverage_ratio` | ICレシオ (倍) | 営業利益 / 営業外費用（支払利息がある場合のみ）|\n\n```python\ns = ratios.safety()\nprint(s.equity_ratio)            # 例: 42.1\nprint(s.interest_coverage_ratio) # 例: 7.3 または None\n```\n\n---\n\n## 効率性指標 — `EfficiencyRatios`\n\n| メソッド | 指標 | 計算式 |\n|----------|------|--------|\n| `asset_turnover` | 総資産回転率 (回) | 売上高 / 総資産 |\n| `inventory_turnover` | 棚卸資産回転率 (回) | 売上原価 / 棚卸資産（棚卸資産>0の場合） |\n| `receivables_turnover` | 売掛金回転率 (回) | 売上高 / 売掛金（売掛金>0の場合） |\n| `days_sales_outstanding` | 売掛金回収日数 (日) | 365 / 売掛金回転率 |\n| `days_inventory_outstanding` | 棚卸資産回転日数 (日) | 365 / 棚卸資産回転率 |\n\n---\n\n## 成長性指標 — `GrowthRatios`\n\n> 前期データ（`prev_income_stmt` / `prev_balance_sheet`）がない場合はすべて `None`\n\n| メソッド | 指標 | 計算式 |\n|----------|------|--------|\n| `revenue_growth_rate` | 売上高成長率 (%) | (当期 - 前期) / |前期| × 100 |\n| `operating_income_growth_rate` | 営業利益成長率 (%) | 同上 |\n| `net_income_growth_rate` | 純利益成長率 (%) | 同上 |\n| `total_assets_growth_rate` | 総資産成長率 (%) | 同上 |\n\n---\n\n## キャッシュフロー指標 — `CashFlowRatios`\n\n| メソッド | 指標 | 計算式 |\n|----------|------|--------|\n| `operating_cf_margin` | 営業CFマージン (%) | 営業CF / 売上高 × 100 |\n| `free_cash_flow` | フリーキャッシュフロー (円) | 営業CF + 設備投資（capexは負値） |\n| `cash_flow_to_debt` | CF / 有利子負債 (倍) | 営業CF / (短期+長期借入金)（借入がある場合のみ） |\n\n---\n\n## ユーティリティメソッド\n\n```python\n# ゼロ除算防止: 分母=0 のとき None を返す\nFinancialRatios._safe_divide(numerator, denominator) -> Optional[float]\n\n# Optional[float] を % 換算 (None は 0.0 扱い)\nFinancialRatios._pct(value) -> float\n```\n\n---\n\n## 使用例\n\n```python\nfrom financial_analysis.ratios import FinancialRatios\n\nratios = FinancialRatios(\n    income_stmt=data.income_statement,\n    balance_sheet=data.balance_sheet,\n    cash_flow=data.cash_flow_statement,\n    prev_income_stmt=data.previous_income_statement,\n)\n\np = ratios.profitability()\ns = ratios.safety()\ne = ratios.efficiency()\ng = ratios.growth()\ncf = ratios.cash_flow_ratios()\n```\n",
    "03_analyzer_report.md": "# 分析エンジン・レポート生成 (`analyzer.py` / `report.py`)\n\n---\n\n## FinancialAnalyzer（分析エンジン）\n\n`financial_analysis/analyzer.py`\n\n### 概要\n\n`FinancialRatios` で計算した各指標に対して、評価コメントと総合スコア（0〜100）を付与します。\n\n### 使い方\n\n```python\nfrom financial_analysis import FinancialAnalyzer\n\nanalyzer = FinancialAnalyzer()\nresult = analyzer.analyze(data)   # FinancialData を渡す\n\nprint(result.overall_score)       # 例: 73.5\nprint(result.overall_comments)    # ['財務状況は概ね健全です...']\n```\n\n### AnalysisResult（結果オブジェクト）\n\n```python\n@dataclass\nclass AnalysisResult:\n    company_name: str\n    period: str\n\n    profitability: ProfitabilityRatios\n    safety: SafetyRatios\n    efficiency: EfficiencyRatios\n    growth: GrowthRatios\n    cash_flow: CashFlowRatios\n\n    profitability_comments: List[str]\n    safety_comments: List[str]\n    efficiency_comments: List[str]\n    growth_comments: List[str]\n    cash_flow_comments: List[str]\n\n    overall_score: Optional[float]   # 0〜100\n    overall_comments: List[str]\n```\n\n### 総合スコア配点\n\n| カテゴリ | 満点 | 主な基準 |\n|----------|------|---------|\n| 収益性 | 30点 | 営業利益率: ≥15%→30点, ≥5%→20点, ≥0%→10点 |\n| 安全性 | 30点 | 自己資本比率(15点) + 流動比率(15点) |\n| 効率性 | 20点 | 総資産回転率: ≥1.5→20点, ≥0.8→13点 |\n| CF | 20点 | FCF≥0 かつ CFマージン≥10%→20点 |\n| **合計** | **100点** | |\n\n### 評価コメントの閾値\n\n**収益性**\n- 売上総利益率: ≥50% 非常に高い / ≥30% 良好 / ≥10% 標準 / <10% 改善必要\n- 営業利益率: ≥15% 優秀 / ≥5% 安定 / ≥0% 要改善 / <0% 損失\n- ROE: ≥15% 高効率 / ≥8% 標準 / <8% 要改善\n\n**安全性**\n- 流動比率: ≥200% 十分 / ≥100% 概ね問題なし / <100% リスクあり\n- 自己資本比率: ≥50% 非常に健全 / ≥30% 健全 / ≥15% 高レバレッジ / <15% 強化急務\n- ICレシオ: ≥5倍 十分 / ≥2倍 余裕少 / <2倍 懸念\n\n**効率性**\n- 総資産回転率: ≥1.5回 効率的 / ≥0.8回 標準 / <0.8回 改善余地\n- 売掛金回収日数: ≤30日 迅速 / ≤60日 標準 / >60日 要改善\n\n**成長性**\n- 売上高成長率: ≥20% 高成長 / ≥5% 安定増収 / ≥0% 横ばい / <0% 減収\n\n**CF**\n- CFマージン: ≥15% 優秀 / ≥5% 安定 / <5% 要改善\n- FCF: ≥0 自己資金調達可 / <0 要管理\n\n---\n\n## ReportGenerator（レポート生成）\n\n`financial_analysis/report.py`\n\n### 使い方\n\n```python\nfrom financial_analysis import ReportGenerator\n\ngenerator = ReportGenerator()\n\n# プレーンテキスト\ntext_report = generator.generate_text(result)\n\n# Markdown\nmd_report = generator.generate_markdown(result)\n\n# ファイル保存\nwith open(\"report.md\", \"w\", encoding=\"utf-8\") as f:\n    f.write(md_report)\n```\n\n### テキストレポートの構成\n\n```\n============================================================\n  財務分析レポート: 株式会社〇〇\n  対象期間: 2024年3月期\n============================================================\n\n総合評価スコア: 73.5 / 100\n  → 財務状況は概ね健全です...\n\n----------------------------------------\n【収益性指標】\n  売上総利益率:    60.00 %\n  営業利益率:      18.00 %\n  ...\n\n【安全性指標】\n【効率性指標】\n【成長性指標】\n【キャッシュフロー指標】\n```\n\n### Markdownレポートの構成\n\n```markdown\n# 財務分析レポート: 株式会社〇〇\n**対象期間:** 2024年3月期\n\n## 総合評価スコア: 73.5 / 100\n> 財務状況は概ね健全です...\n\n## 収益性指標\n| 指標 | 値 |\n|------|-----|\n| 売上総利益率 | 60.00% |\n...\n\n## 安全性指標\n## 効率性指標\n## 成長性指標\n## キャッシュフロー指標\n```\n",
    "04_loaders.md": "# ファイルローダー群\n\n対応形式: **PDF** / **Excel (.xlsx .xls .xlsm)** / **Word (.docx)** / **Google Drive**\n\n---\n\n## 統合ローダー — `load_financial_data`\n\n`financial_analysis/file_loader.py`\n\n拡張子を自動判定し、適切なローダーに振り分けます。\n\n```python\nfrom financial_analysis import load_financial_data\n\ndata = load_financial_data(\n    file_path,                # 必須: ファイルパス\n    company_name=\"株式会社〇〇\",   # 省略時: ファイル名\n    period=\"2024年3月期\",          # 省略時: \"\"\n    unit=1.0,                      # 単位倍率（千円=1000, 百万円=1000000）\n    sheet_name=None,               # Excelのシート名（省略時: 自動選択）\n)\n```\n\n### デバッグ確認\n\n```python\nfrom financial_analysis import extract_debug_text\n\n# 生テキスト・表構造を文字列で取得\ndebug_str = extract_debug_text(\"決算書.pdf\")\nprint(debug_str)\n```\n\n---\n\n## PDFローダー\n\n`financial_analysis/pdf_loader.py` — PyMuPDF (fitz) を使用\n\n### 数値認識\n\n- 通常数値: `1,234,567`\n- △▲ 表記の負数: `△500` → `-500`\n- カッコ表記の負数: `(500)` → `-500`\n\n### テーブル解析パターン\n\n| パターン | 形式 | 説明 |\n|----------|------|------|\n| A | 科目が行・月が列 | 合計列を優先使用 |\n| B | 月が行・科目が列 | 月を集計 |\n| C | 2列（科目 / 金額） | 座標ベース抽出の結果 |\n\n### 座標ベース抽出\n\n日本語PDFではラベルと数値が別テキストブロックに分離することがあるため、  \nx座標でラベル（左列）と数値（右列）を照合して自動結合します。\n\n### キーワードマップ（抜粋）\n\n| 日本語 | フィールド |\n|--------|-----------|\n| 売上高, 営業収益, 売上収入 | `revenue` |\n| 売上原価, 製造原価, 原価 | `cost_of_goods_sold` |\n| 売上総利益, 粗利 | `gross_profit` |\n| 販売費, 販管費 | `selling_expenses` |\n| 営業利益, 事業利益 | `operating_income` |\n| 当期純利益, 最終利益, 純損益 | `net_income` |\n\n---\n\n## Excelローダー\n\n`financial_analysis/excel_loader.py` — openpyxl を使用\n\n### シート自動選択\n\nシート名に **損益 / PL / P&L / 売上** を含むシートを優先選択。\n\n### テーブル解析パターン\n\n| パターン | 形式 |\n|----------|------|\n| A | 科目が行、月が列（合計列を自動検出） |\n| B | 月が行、科目が列 |\n\n### 合計列の自動検出\n\n- 「合計」「計」「累計」「Total」などを含む列\n- 月数値列の最大値を持つ列\n\n### シート名を指定する場合\n\n```python\ndata = load_financial_data(\"月次PL.xlsx\", sheet_name=\"損益計算書\")\n```\n\n---\n\n## Wordローダー\n\n`financial_analysis/word_loader.py` — python-docx を使用\n\n### 解析手順\n\n1. **表（Table）優先**: ドキュメント内のすべてのテーブルを走査\n2. **段落テキスト（フォールバック）**: テーブルがない場合は本文テキストを解析\n\n### 注意\n\n- `.docx` のみ対応。古い `.doc` 形式は対応不可。\n- `.doc` を指定した場合、`ValueError` が発生し変換方法を案内します。\n\n---\n\n## Google Driveローダー\n\n`financial_analysis/google_drive_loader.py`\n\n### 対応URLパターン\n\n| URLの形式 | ダウンロード形式 |\n|-----------|----------------|\n| `drive.google.com/file/d/{ID}` | ファイルをそのままダウンロード |\n| `docs.google.com/spreadsheets/d/{ID}` | Excel形式でエクスポート |\n| `docs.google.com/document/d/{ID}` | Word形式でエクスポート |\n\n### 公開ファイル（サービスアカウント不要）\n\n```python\nfrom financial_analysis import GoogleDriveLoader\n\nloader = GoogleDriveLoader()\ndata = loader.load_from_url(\n    \"https://drive.google.com/file/d/xxxxx/view?usp=sharing\",\n    company_name=\"株式会社〇〇\",\n    period=\"2024年3月期\",\n    unit=1.0,\n)\n```\n\n### 非公開ファイル（サービスアカウント）\n\n```python\nloader = GoogleDriveLoader(credentials_path=\"service_account.json\")\ndata = loader.load_from_url(\"https://drive.google.com/file/d/xxxxx/view\")\n```\n\n### 注意\n\n- プロキシ環境（社内ネットワーク等）では 403 エラーが発生することがあります。\n- その場合はファイルをローカルにダウンロードして `load_financial_data()` に渡してください。\n- **Google Colab を使用すると、Drive を直接マウントできるためプロキシ問題を回避できます。**\n",
    "05_interfaces.md": "# 実行インターフェース\n\n財務分析ツールには 3 つの実行方法があります。\n\n---\n\n## 1. CLIスクリプト (`examples/run_analysis.py`)\n\nターミナル / コマンドプロンプトから実行する方法です。\n\n### 基本的な使い方\n\n```bash\n# サンプルデータで実行（ファイル不要）\npython examples/run_analysis.py\n\n# ローカルファイルを指定\npython examples/run_analysis.py --input 決算書.pdf\npython examples/run_analysis.py --input 月次PL.xlsx\npython examples/run_analysis.py --input 財務報告.docx\n\n# Google Drive URL を指定\npython examples/run_analysis.py --gdrive \"https://drive.google.com/file/d/xxxxx/view?usp=sharing\"\n```\n\n### オプション一覧\n\n| オプション | 短縮 | 説明 | デフォルト |\n|-----------|------|------|-----------|\n| `--input FILE` | `-i` | ローカルファイルのパス | — |\n| `--gdrive URL` | `-g` | Google Drive 共有リンク | — |\n| `--credentials JSON` | — | サービスアカウントキーJSON | — |\n| `--company NAME` | — | 企業名 | ファイル名 |\n| `--period PERIOD` | — | 会計期間（例: 2024年3月期） | \"\" |\n| `--unit MULTIPLIER` | — | 金額単位倍率（千円=1000） | 1.0 |\n| `--sheet SHEET_NAME` | — | Excelのシート名 | 自動選択 |\n| `--format` | — | `text` or `markdown` | text |\n| `--output FILE` | — | 出力ファイルパス（省略=標準出力） | 標準出力 |\n| `--debug` | — | 抽出テキスト・表を表示 | — |\n\n### 実行例\n\n```bash\n# 千円単位のExcelを指定し、Markdownで保存\npython examples/run_analysis.py \\\n    --input 月次PL.xlsx \\\n    --company \"株式会社〇〇\" \\\n    --period \"2024年3月期\" \\\n    --unit 1000 \\\n    --format markdown \\\n    --output report.md\n\n# 抽出確認（科目が正しく認識されているか確認）\npython examples/run_analysis.py --input 決算書.pdf --debug\n\n# Googleスプレッドシート（サービスアカウント不要の公開ファイル）\npython examples/run_analysis.py \\\n    --gdrive \"https://docs.google.com/spreadsheets/d/xxxxx/edit\"\n```\n\n---\n\n## 2. Streamlit Web UI (`app.py`)\n\nブラウザからファイルをドラッグ＆ドロップして分析する方法です。  \n**PCでの利用を推奨**（モバイル・タブレットでは操作性が限られます）。\n\n### 起動方法\n\n```bash\npip install streamlit\nstreamlit run app.py\n# ブラウザが自動的に http://localhost:8501 を開きます\n```\n\n### 機能\n\n- PDF / Excel (.xlsx .xls .xlsm) / Word (.docx) のアップロード\n- サイドバーで企業名・会計期間・金額単位・シート名を設定\n- 抽出した主要数値（5項目）のメトリクス表示\n- タブ別の詳細指標表示（収益性 / 安全性 / 効率性 / 成長性 / CF）\n- テキスト or Markdown 形式でレポートをダウンロード\n\n### 画面構成\n\n```\n[サイドバー]           [メインエリア]\n・企業名               ファイルアップロードエリア\n・会計期間             ─────────────────────\n・金額単位             抽出結果サマリー（5指標）\n・シート名             ─────────────────────\n・レポート形式         総合評価スコア + プログレスバー\n                      タブ: 収益性/安全性/効率性/成長性/CF\n                      ─────────────────────\n                      レポートダウンロードボタン\n```\n\n---\n\n## 3. Google Colab ノートブック (`financial_analysis_colab.ipynb`)\n\nタブレット・スマートフォンでも利用可能。Google Drive のファイルを直接分析できます。\n\n### セル構成\n\n| セル | 内容 |\n|------|------|\n| ① | ライブラリインストール (`pymupdf`, `openpyxl`, `python-docx`) |\n| ② | 財務分析コード全文（inline） |\n| ③ | Google Drive マウント |\n| ④ | **ファイルマネージャー** — `scan_drive()` でDrive内を検索・番号表示 |\n| ⑤ | `FILE_NUMBER` で1ファイル選択・分析 |\n| ⑥ | `FILE_NUMBERS` リストで複数ファイル一括分析 |\n| ⑦ | デバッグ確認（シート名・先頭行・テキスト） |\n\n### ファイルマネージャーの使い方（セル④）\n\n```\nセル④を実行すると:\n\n📄  1. 月次PL_2024.pdf        /content/drive/MyDrive/財務/月次PL_2024.pdf\n📊  2. 決算書.xlsx             /content/drive/MyDrive/決算/決算書.xlsx\n📝  3. 財務報告書.docx         /content/drive/MyDrive/報告/財務報告書.docx\n\n合計 3 件のファイルが見つかりました\n```\n\n```python\n# セル⑤: 番号で選択\nFILE_NUMBER = 2       # → 決算書.xlsx を分析\nCOMPANY_NAME = \"株式会社〇〇\"\nPERIOD = \"2024年3月期\"\nUNIT = 1.0\n```\n\n```python\n# セル⑥: 複数ファイルを一括分析\nFILE_NUMBERS = [1, 2, 3]\n```\n\n### Colabを開く手順\n\n1. [Google Colab](https://colab.research.google.com/) にアクセス\n2. 「ファイル」→「ノートブックをアップロード」→ `financial_analysis_colab.ipynb` を選択\n3. または GitHub URL から直接開く\n\n---\n\n## 方法の選択ガイド\n\n| 状況 | 推奨方法 |\n|------|---------|\n| PC でターミナルが使える | CLI スクリプト |\n| PC でブラウザから使いたい | Streamlit Web UI |\n| タブレット / スマートフォン | Google Colab ノートブック |\n| Google Drive のファイルを分析 | Colab（Driveマウント）|\n| バッチ処理・自動化 | CLI スクリプト |\n| 複数ファイルをまとめて分析 | Colab（セル⑥）|\n",
    "06_architecture.md": "# アーキテクチャ概要\n\n---\n\n## ディレクトリ構成\n\n```\nプロジェクトルート/\n├── financial_analysis/          # コアライブラリ\n│   ├── __init__.py              # 公開API定義\n│   ├── models.py                # データモデル（dataclass）\n│   ├── ratios.py                # 財務比率計算\n│   ├── analyzer.py              # 評価・スコアリング\n│   ├── report.py                # レポート生成（テキスト / Markdown）\n│   ├── file_loader.py           # 統合ローダー（拡張子判定）\n│   ├── pdf_loader.py            # PDFローダー（PyMuPDF）\n│   ├── excel_loader.py          # Excelローダー（openpyxl）\n│   ├── word_loader.py           # Wordローダー（python-docx）\n│   └── google_drive_loader.py   # Google Drive ローダー\n│\n├── examples/\n│   ├── run_analysis.py          # CLIスクリプト\n│   ├── sample_data.py           # サンプル財務データ\n│   ├── test_income_statement.pdf\n│   ├── test_monthly_pl.xlsx\n│   └── test_monthly_pl.docx\n│\n├── tests/\n│   ├── test_models.py\n│   ├── test_ratios.py\n│   └── test_analyzer.py\n│\n├── docs/                        # ドキュメント（このフォルダ）\n│\n├── app.py                       # Streamlit Web UI\n├── financial_analysis_colab.ipynb  # Google Colab ノートブック\n└── requirements.txt\n```\n\n---\n\n## データフロー\n\n```\nファイル（PDF/Excel/Word/Google Drive）\n        │\n        ▼\n  load_financial_data()          ← file_loader.py（拡張子で振り分け）\n        │\n        ├── PdfIncomeStatementLoader      (PyMuPDF)\n        ├── ExcelIncomeStatementLoader    (openpyxl)\n        ├── WordIncomeStatementLoader     (python-docx)\n        └── GoogleDriveLoader             (requests / google-api)\n        │\n        ▼\n  FinancialData                  ← models.py\n    ├── IncomeStatement\n    ├── BalanceSheet\n    ├── CashFlowStatement\n    └── (前期データ)\n        │\n        ▼\n  FinancialRatios                ← ratios.py\n    ├── profitability()\n    ├── safety()\n    ├── efficiency()\n    ├── growth()\n    └── cash_flow_ratios()\n        │\n        ▼\n  FinancialAnalyzer.analyze()    ← analyzer.py\n    → AnalysisResult\n      ├── 各指標値\n      ├── 評価コメント\n      └── 総合スコア (0〜100)\n        │\n        ▼\n  ReportGenerator                ← report.py\n    ├── generate_text()          → .txt\n    └── generate_markdown()      → .md\n```\n\n---\n\n## 依存ライブラリ\n\n| ライブラリ | 用途 | バージョン |\n|-----------|------|-----------|\n| `pymupdf` | PDF テキスト・座標抽出 | ≥1.23 |\n| `openpyxl` | Excel (.xlsx) 読み込み | ≥3.1 |\n| `python-docx` | Word (.docx) 読み込み | ≥1.0 |\n| `requests` | Google Drive 公開ファイルDL | ≥2.31 |\n| `google-api-python-client` | Google Drive API（サービスアカウント） | ≥2.0 |\n| `google-auth` | Google 認証 | ≥2.0 |\n| `streamlit` | Web UI | ≥1.30 |\n| `pytest` | テスト実行 | ≥7.0 |\n\n---\n\n## 公開API（`financial_analysis/__init__.py`）\n\n```python\nfrom financial_analysis import (\n    # データモデル\n    IncomeStatement,\n    BalanceSheet,\n    CashFlowStatement,\n    FinancialData,\n\n    # 比率計算\n    FinancialRatios,\n\n    # 分析・レポート\n    FinancialAnalyzer,\n    ReportGenerator,\n\n    # ローダー\n    PdfIncomeStatementLoader,\n    ExcelIncomeStatementLoader,\n    WordIncomeStatementLoader,\n    GoogleDriveLoader,\n\n    # ユーティリティ\n    load_financial_data,\n    extract_debug_text,\n)\n```\n\n---\n\n## テスト\n\n```bash\n# 全テスト実行\npytest tests/ -v\n\n# 個別ファイル\npytest tests/test_models.py -v\npytest tests/test_ratios.py -v\npytest tests/test_analyzer.py -v\n```\n\nテストファイル一覧:\n\n| ファイル | 内容 |\n|---------|------|\n| `test_models.py` | dataclassの自動計算ロジック |\n| `test_ratios.py` | 財務比率の計算精度 |\n| `test_analyzer.py` | 評価コメント・スコアのロジック |\n",
    "skill.md": "# 財務分析ツール — 機能・技術スキル一覧\n\nこのドキュメントはClaude Projectのカスタム手順補足用です。  \nツールが「できること」と「使っている技術」をまとめています。\n\n---\n\n## できること（機能一覧）\n\n### ファイル読み込み\n\n| 形式 | 拡張子 | 特記事項 |\n|------|--------|---------|\n| PDF | `.pdf` | テキストPDFのみ（スキャン不可）。△▲/(xxx)の負数表記に対応 |\n| Excel | `.xlsx` `.xls` `.xlsm` | シート自動選択。千円・百万円単位に対応 |\n| Word | `.docx` | 表形式優先、段落テキストにもフォールバック |\n| Google Drive | 共有URL | PDF/スプレッドシート/ドキュメントのURLを直接指定可 |\n\n### 財務指標の自動計算\n\n| カテゴリ | 指標 |\n|----------|------|\n| 収益性 | 売上総利益率・営業利益率・純利益率・ROA・ROE・EBITDAマージン |\n| 安全性 | 流動比率・当座比率・自己資本比率・負債資本比率・ICレシオ |\n| 効率性 | 総資産回転率・棚卸資産回転率・売掛金回収日数 |\n| 成長性 | 売上高・営業利益・純利益・総資産の各成長率（前期比較） |\n| CF | 営業CFマージン・フリーキャッシュフロー・CF/有利子負債 |\n\n### 評価・レポート\n\n- 各指標に対してコメントを自動生成（★評価付き）\n- 総合スコア（0〜100点）を算出\n- テキスト形式 / Markdown形式でレポート出力\n- ファイルへの保存またはダウンロード\n\n### 実行環境\n\n| 方法 | 対象ユーザー | 特徴 |\n|------|-------------|------|\n| Google Colab | タブレット・スマホ | Drive直接マウント、番号選択でファイル指定 |\n| Streamlit WebUI | PC | ブラウザからドラッグ&ドロップ |\n| CLIスクリプト | PC（ターミナル） | バッチ処理・自動化向け |\n\n---\n\n## 技術スタック\n\n### 言語・フレームワーク\n\n| 技術 | バージョン | 用途 |\n|------|-----------|------|\n| Python | 3.9以上 | メイン言語 |\n| dataclasses | 標準ライブラリ | データモデル定義・自動計算 |\n| streamlit | ≥1.30 | Web UI |\n| pytest | ≥7.0 | 単体テスト（27件） |\n\n### ライブラリ\n\n| ライブラリ | バージョン | 用途 |\n|-----------|-----------|------|\n| PyMuPDF (fitz) | ≥1.23 | PDF テキスト・座標抽出 |\n| openpyxl | ≥3.1 | Excel (.xlsx) 読み込み |\n| python-docx | ≥1.0 | Word (.docx) 読み込み |\n| requests | ≥2.31 | Google Drive 公開ファイルDL |\n| google-api-python-client | ≥2.0 | Google Drive API（サービスアカウント）|\n| google-auth | ≥2.0 | Google 認証 |\n\n### 設計パターン\n\n| パターン | 適用箇所 |\n|----------|---------|\n| dataclass + `__post_init__` | 財務諸表の自動計算（models.py） |\n| Strategy パターン | ローダーの拡張子別切り替え（file_loader.py） |\n| ゼロ除算ガード | `_safe_divide()` + `_pct()` ヘルパー（ratios.py） |\n| 座標ベース抽出 | 日本語PDFのラベル・数値マッチング（pdf_loader.py） |\n| 定数マップ | `KEYWORD_MAP` で日本語科目名を内部フィールドに変換 |\n\n---\n\n## ソースコード構成\n\n```\nfinancial_analysis/\n├── models.py          # IncomeStatement / BalanceSheet / CashFlowStatement\n├── ratios.py          # 財務比率計算（5カテゴリ）\n├── analyzer.py        # 評価コメント + 総合スコア（0〜100）\n├── report.py          # テキスト / Markdown レポート生成\n├── file_loader.py     # 統合ローダー（拡張子で自動振り分け）\n├── pdf_loader.py      # PyMuPDF ベースのPDFローダー\n├── excel_loader.py    # openpyxl ベースのExcelローダー\n├── word_loader.py     # python-docx ベースのWordローダー\n└── google_drive_loader.py  # Google Drive URL対応ローダー\n```\n\n---\n\n## 科目名認識キーワード（主要）\n\nファイル内の科目名がこれらのいずれかと一致・部分一致すれば自動認識されます。\n\n| 科目 | 認識されるキーワード |\n|------|---------------------|\n| 売上高 | 売上高、売上、営業収益、売上収入、収益合計 |\n| 売上原価 | 売上原価、製造原価、原価合計 |\n| 売上総利益 | 売上総利益、粗利、粗利益 |\n| 販管費 | 販売費及び一般管理費、販管費、販売費、一般管理費 |\n| 営業利益 | 営業利益、営業損益、営業損失 |\n| 経常利益 | 経常利益、経常損益、経常損失 |\n| 当期純利益 | 当期純利益、当期純損失、当期利益、純利益 |\n| 法人税等 | 法人税、法人税等、法人税・住民税及び事業税 |\n\n---\n\n## 既知の制限\n\n| 制限 | 内容 |\n|------|------|\n| スキャンPDF不可 | 画像PDFは認識できません。テキスト選択できるPDFが対象 |\n| `.doc`非対応 | 旧Word形式。`.docx`に変換して使用 |\n| プロキシ環境 | 社内ネット等からのGoogle Drive直接DLは403エラーになる場合あり |\n| 前期比較 | 成長率は前期データを手動で渡した場合のみ算出可能 |\n| 貸借対照表・CF | Colabノートブック版では損益計算書のみ対応（ライブラリ版は全対応）|\n",
    "SETUP_GUIDE.md": "# 財務分析ツール — セットアップ・操作手順書\n\n---\n\n## 目次\n\n1. [環境準備](#1-環境準備)\n2. [インストール](#2-インストール)\n3. [Colabで使う（タブレット推奨）](#3-google-colabで使うタブレット推奨)\n4. [CLIで使う（PC）](#4-cliで使うpc)\n5. [WebUIで使う（PC）](#5-webui-streamlitで使うpc)\n6. [ファイルが正しく読み込めない場合](#6-ファイルが正しく読み込めない場合)\n7. [対応ファイル形式と準備のポイント](#7-対応ファイル形式と準備のポイント)\n\n---\n\n## 1. 環境準備\n\n### Colabを使う場合（タブレット・PC）\n\n- Googleアカウントがあれば追加インストール不要\n- 手順3へ進んでください\n\n### ローカル（PC）で動かす場合\n\n- Python 3.9 以上\n- pip が使用可能なこと\n\n---\n\n## 2. インストール\n\n```bash\n# リポジトリのクローン（またはダウンロード）\ngit clone <リポジトリURL>\ncd <プロジェクトフォルダ>\n\n# 依存パッケージのインストール\npip install -r requirements.txt\n```\n\n`requirements.txt` の内容:\n\n```\npytest>=7.0\npymupdf>=1.23\nopenpyxl>=3.1\npython-docx>=1.0\nrequests>=2.31\ngoogle-api-python-client>=2.0\ngoogle-auth>=2.0\nstreamlit>=1.30\n```\n\n---\n\n## 3. Google Colabで使う（タブレット推奨）\n\nタブレットやスマートフォンでも Google Drive のファイルを直接分析できます。\n\n### 手順\n\n#### ステップ1: Colabを開く\n\n1. ブラウザで [https://colab.research.google.com/](https://colab.research.google.com/) にアクセス\n2. 「ファイル」→「ノートブックをアップロード」\n3. `financial_analysis_colab.ipynb` を選択してアップロード\n\n> Googleドライブに保存済みの場合は「ドライブ」タブからも開けます。\n\n#### ステップ2: ライブラリをインストール（セル①）\n\n```python\n# セル①を実行（▶ボタンをタップ）\n!pip install pymupdf openpyxl python-docx\n```\n\n> **初回のみ実行** が必要です。ランタイムが変わると再実行が必要になります。\n\n#### ステップ3: コードを読み込む（セル②）\n\nセル②を実行します。財務分析コードがすべてメモリに読み込まれます。\n\n#### ステップ4: Google Driveをマウント（セル③）\n\n```python\n# セル③を実行\nfrom google.colab import drive\ndrive.mount('/content/drive')\n```\n\n1. 実行すると認証リンクが表示される\n2. リンクをタップしてGoogleアカウントにログイン\n3. 表示されたコードをコピーして入力ボックスに貼り付け\n4. 「ドライブがマウントされました」と表示されれば完了\n\n#### ステップ5: ファイルマネージャーを起動（セル④）\n\nセル④をそのまま実行してください。Drive 内の対応ファイルが番号付きで一覧表示されます。  \n出力例は [`docs/05_interfaces.md`](./05_interfaces.md#3-google-colabノートブック) を参照。\n\n> `SEARCH_DEPTH` の値を増やすと深い階層まで検索します（デフォルト: 3）\n\n#### ステップ6: ファイルを選択して分析（セル⑤）\n\n```python\nFILE_NUMBER = 2          # 分析したいファイルの番号\nCOMPANY_NAME = \"株式会社〇〇\"   # 企業名（省略可）\nPERIOD = \"2024年3月期\"   # 会計期間（省略可）\nUNIT = 1.0               # 金額単位（千円=1000, 百万円=1000000）\nSHEET_NAME = None        # Excelのシート名（省略=自動選択）\n```\n\nセル⑤を実行すると分析結果がノートブック上に表示されます。\n\n#### ステップ7（オプション）: 複数ファイルを一括分析（セル⑥）\n\n```python\nFILE_NUMBERS = [1, 2, 3]   # 分析したいファイルの番号リスト\n```\n\n---\n\n## 4. CLIで使う（PC）\n\n### サンプルデータで試す\n\n```bash\npython examples/run_analysis.py\n```\n\n### ローカルファイルを分析\n\n```bash\n# PDF\npython examples/run_analysis.py --input 決算書.pdf\n\n# Excel（千円単位、シート名指定）\npython examples/run_analysis.py \\\n    --input 月次PL.xlsx \\\n    --company \"株式会社〇〇\" \\\n    --period \"2024年3月期\" \\\n    --unit 1000 \\\n    --sheet \"損益計算書\"\n\n# Word\npython examples/run_analysis.py --input 財務報告.docx\n```\n\n### レポートをファイルに保存\n\n```bash\n# Markdownで保存\npython examples/run_analysis.py \\\n    --input 決算書.pdf \\\n    --format markdown \\\n    --output report.md\n\n# テキストで保存\npython examples/run_analysis.py \\\n    --input 決算書.pdf \\\n    --output report.txt\n```\n\n### Google Drive から分析\n\n```bash\n# 公開ファイル（共有リンクが有効なファイル）\npython examples/run_analysis.py \\\n    --gdrive \"https://drive.google.com/file/d/xxxxx/view?usp=sharing\"\n\n# 非公開ファイル（サービスアカウントが必要）\npython examples/run_analysis.py \\\n    --gdrive \"https://drive.google.com/file/d/xxxxx/view\" \\\n    --credentials service_account.json\n```\n\n---\n\n## 5. WebUI (Streamlit)で使う（PC）\n\n### 起動\n\n```bash\nstreamlit run app.py\n```\n\nブラウザが自動的に `http://localhost:8501` を開きます。\n\n### 操作手順\n\n1. **サイドバー**で設定を入力:\n   - 企業名（省略可）\n   - 会計期間（省略可）\n   - 金額単位（円 / 千円 / 百万円）\n   - Excelシート名（省略可）\n   - レポート形式（テキスト / Markdown）\n\n2. **ファイルアップロードエリア**にファイルをドラッグ＆ドロップ、またはクリックして選択\n   - 対応形式: PDF / Excel (.xlsx .xls) / Word (.docx)\n\n3. 分析結果が自動表示されます:\n   - 抽出した主要数値（5項目）\n   - 総合評価スコア\n   - タブ別の詳細指標（収益性 / 安全性 / 効率性 / 成長性 / CF）\n\n4. **レポートをダウンロード**ボタンでファイル保存\n\n---\n\n## 6. ファイルが正しく読み込めない場合\n\n### 売上高が 0 になる\n\n科目名が標準的な表記と異なる可能性があります。\n\n**CLIでデバッグ確認:**\n\n```bash\npython examples/run_analysis.py --input 決算書.pdf --debug\n```\n\n**Colabでデバッグ確認（セル⑦）:**\n\nセル⑦を実行すると、シート名・先頭行・抽出テキストが表示されます。\n\n**対処方法:**\n- ファイル内の科目名を確認し、認識可能な表記に合わせる。  \n  認識可能なキーワード一覧は [`docs/04_loaders.md`](./04_loaders.md#キーワードマップ抜粋) を参照。\n\n### Excelで複数シートがある\n\n```bash\n# シート名を明示指定\npython examples/run_analysis.py --input 月次PL.xlsx --sheet \"損益計算書\"\n```\n\nStreamlit の場合はサイドバーの「Excelシート名」欄に入力。\n\n### PDFの数値が認識されない\n\n- スキャンPDF（画像PDF）は対応していません。テキスト選択ができるPDFが必要です。\n- △▲ や (xxx) 形式の負数表記は自動的に負値に変換されます。\n\n### `.doc` 形式のWordファイル\n\n`.doc` は非対応です。Word で開き「名前を付けて保存」→「.docx 形式」で保存してください。\n\n### Google Drive にアクセスできない\n\nプロキシ環境では接続が遮断されることがあります。  \n→ ファイルをローカルにダウンロードし、`--input` オプションで指定してください。  \n→ またはGoogle Colabを使用すると（Drive直接マウント）プロキシ問題を回避できます。\n\n---\n\n## 7. 対応ファイル形式と準備のポイント\n\n### PDF\n\n- テキストPDF（コピー＆ペーストができるもの）が対象\n- 1ページに収まる損益計算書が最も精度が高い\n- 縦書きや特殊フォントは認識精度が下がることがあります\n\n### Excel (.xlsx / .xls / .xlsm)\n\n| 推奨レイアウト | 説明 |\n|---------------|------|\n| **パターンA**（推奨） | 科目が行、月が列（合計列あり） |\n| パターンB | 月が行、科目が列 |\n\n- 先頭行や先頭列にタイトルがある場合も自動スキップします\n- 金額単位が「千円」の場合は `--unit 1000` を指定してください\n\n### Word (.docx)\n\n- 表形式（Wordのテーブル）のデータを優先して読み込みます\n- 箇条書き・段落のみのドキュメントでもフォールバック解析します\n\n---\n\n> CLI コマンドの詳細なオプションと実行例は [セクション4](#4-cliで使うpc) を参照してください。\n"
}

os.makedirs('docs', exist_ok=True)
for filename, content in DOCS.items():
    path = f'docs/{filename}'
    with open(path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f'✅ {filename} を保存しました')

# ZIPにまとめてダウンロード
import zipfile
zip_path = '財務分析ツール_ドキュメント.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for filename in DOCS:
        zf.write(f'docs/{filename}', filename)
print(f'\n📦 ZIPファイルを作成しました: {zip_path}')
files.download(zip_path)
print('⬇️ ダウンロードを開始しました')